### Data Preprocessing

#### Data Cleaning

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('Sikorsky_wind.csv', low_memory=False)
df['TmStamp'] = pd.to_datetime(df['TmStamp'], format='mixed')
df['WSPD'] = pd.to_numeric(df['WSPD'], errors='coerce')
df['WDIR'] = pd.to_numeric(df['WDIR'], errors='coerce')

# In total 123,528 hourly timestamps
date_ranges = [
    ('2004-11-01 00:00:00', '2015-01-05 23:00:00'),  # 89232 hourly timestamps
    ('2016-12-12 00:00:00', '2018-01-11 23:00:00'),  # 9504 hourly timestamps
    ('2018-04-27 00:00:00', '2018-10-22 23:00:00'),  # 4296 hourly timestamps
    ('2019-05-28 00:00:00', '2019-12-31 23:00:00'),  # 5232 hourly timestamps
    ('2022-12-09 00:00:00', '2023-06-02 23:00:00'),  # 4224 hourly timestamps
    ('2024-03-28 00:00:00', '2025-06-30 23:00:00')]  # 11040 hourly timestamps

# Filter date ranges
ndf_wind = pd.DataFrame()
for start, end in [(pd.to_datetime(start), pd.to_datetime(end[:-5] + '59:59')) for start, end in date_ranges]:
    ndf_wind = pd.concat([ndf_wind, df[(df['TmStamp'] >= start) & (df['TmStamp'] <= end)]])

# Remove outliers
ndf_wind.loc[ndf_wind['WSPD'] < 0.0, 'WSPD'] = np.nan
ndf_wind.loc[ndf_wind['WDIR'] == 999.0, 'WDIR'] = np.nan

# Resample by hour
ndf_wind['TmStamp'] = ndf_wind['TmStamp'].dt.floor('h')
ndf_wind = ndf_wind.loc[ndf_wind.groupby('TmStamp')['WSPD'].idxmax().dropna()]

# Fill in the missing timestamps
timestamps = []
for start, end in [(pd.to_datetime(start), pd.to_datetime(end)) for start, end in date_ranges]:
    timestamps.extend(pd.date_range(start, end, freq='h'))

# Merge datasets
df_new = pd.merge(pd.DataFrame({'TmStamp':timestamps}), ndf_wind, on='TmStamp', how='left')
df_new = df_new.sort_values('TmStamp').reset_index(drop=True)

#### Data Imputation

In [2]:
# Wind data imputation
# Count consecutive NaN lengths
def get_nan_lengths(series):
    nans = series.isna()
    group = (nans != nans.shift()).cumsum()
    nan_lengths = nans.groupby(group).transform('sum')
    return np.where(nans, nan_lengths, 0).astype(int)

# Case 1: Interpolate when nan_lengths ≤ 5
WSPD = df_new['WSPD'].interpolate(method='linear')
WDIR = df_new['WDIR'].interpolate(method='linear')

n_random = np.random.normal(loc=0, scale=1, size=len(df_new))
sd_WSPD = np.std(df_new['WSPD'].dropna())
WSPD = WSPD + n_random * sd_WSPD

WSPD_nans = get_nan_lengths(df_new['WSPD'])
WDIR_nans = get_nan_lengths(df_new['WDIR'])
WSPD_mask = (WSPD_nans > 0) & (WSPD_nans <= 5)
WDIR_mask = (WDIR_nans > 0) & (WDIR_nans <= 5)

df_new.loc[WSPD_mask, 'WSPD'] = WSPD[WSPD_mask]
df_new.loc[WDIR_mask, 'WDIR'] = WDIR[WDIR_mask]

In [3]:
# Wind data imputation
# Case 2: Sample from known ('WSPD', 'WDIR') when nan_lengths > 5
df_new['season'] = np.where(df_new['TmStamp'].dt.month.isin([11, 12, 1, 2, 3]), 'windy', 'calm')
for season in ['windy','calm']:
    df_season = df_new[df_new['season'] == season]
    mask = (WDIR_nans > 5) & (df_new['season'] == season)
    sampled = df_season[['WSPD','WDIR']].dropna().sample(n=mask.sum(), replace=True)
    df_new.loc[mask, ['WSPD','WDIR']] = sampled.values

#### Processed Dataset Output

In [4]:
df_new['WSPD'] = df_new['WSPD'].clip(lower=0.25)
df_new['WDIR'] = np.round(df_new['WDIR'] / 10) * 10

# Save the processed dataset
df_new = df_new[['TmStamp','WSPD','WDIR']]
df_new = df_new.sort_values('TmStamp').reset_index(drop=True)
df_new.to_csv('Sikorsky_data.csv', index=False)

df_new.head()

,TmStamp,WSPD,WDIR
0,2004-11-01 00:00:00,5.1,290.0
1,2004-11-01 01:00:00,5.2,290.0
2,2004-11-01 02:00:00,3.1,290.0
3,2004-11-01 03:00:00,3.1,280.0
4,2004-11-01 04:00:00,3.6,290.0
